# Pipeline AURUM — demostracion end-to-end

Ejecuta los cuatro componentes del Ejercicio A-1 sobre el extracto completo `OP_AURUM_extract.csv`
y muestra que hace cada uno: cuantos centinelas de ley se imputaron y cuantos quedaron marcados,
como codifica el objetivo cada categorica, que features aparecen y con que cobertura, y la tabla
final consolidada.

El analisis que justifica cada decision de diseno esta en
[`../exploration/eda_opus.ipynb`](../exploration/eda_opus.ipynb). Este notebook no vuelve a
argumentar: ejecuta y verifica.

**Ejecucion.** Requiere el extracto, que no se versiona. Se busca en `AURUM_CSV_PATH`, luego en
`data/` del repositorio y por ultimo en un `insumos/` hermano. Desde la raiz del repositorio:

```
jupyter nbconvert --to notebook --execute --inplace modulo_a/aurum_pipeline/pipeline_demo.ipynb
```

In [1]:
"""Preparacion: rutas, logging y carga del extracto."""

from __future__ import annotations

import logging
import os
import sys
import time
from collections.abc import Callable
from pathlib import Path

import pandas as pd

RAIZ = next(padre for padre in Path.cwd().resolve().parents if (padre / "modulo_a").exists()) \
    if not (Path.cwd() / "modulo_a").exists() else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ / "modulo_a"))

from aurum_pipeline import (  # noqa: E402
    AurumFeatureBuilder,
    AurumImputer,
    AurumShiftEncoder,
    AurumTransformer,
    domain,
)

# El paquete no configura logging: lo hace quien lo usa. Aqui se activa en INFO para poder
# leer el registro por frente_id que exige el enunciado.
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(message)s", force=True)
logging.getLogger("aurum_pipeline").setLevel(logging.INFO)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)


def localizar_extracto(nombre: str = "OP_AURUM_extract.csv") -> Path:
    """Ubica el extracto sin escribir rutas absolutas en el notebook."""
    del_entorno = os.environ.get("AURUM_CSV_PATH")
    if del_entorno:
        return Path(del_entorno).expanduser().resolve()
    for candidato in (RAIZ / "data" / nombre, RAIZ.parent / "insumos" / nombre):
        if candidato.exists():
            return candidato.resolve()
    raise FileNotFoundError(f"No se encontro {nombre}. Copialo en data/ o define AURUM_CSV_PATH.")


RUTA = localizar_extracto()
crudo = pd.read_csv(RUTA, parse_dates=[domain.COLUMNA_TIEMPO]).sort_values(domain.COLUMNA_TIEMPO)

print(f"origen  : {RUTA}")
print(f"filas   : {len(crudo):,}")
print(f"columnas: {crudo.shape[1]}")

origen  : /Users/amadorcallelo/projects/personal/pruebas_tecnicas/mineros/data_scientist_senior/repositorio-ds-mine/data/OP_AURUM_extract.csv
filas   : 50,000
columnas: 18


## 1. El pipeline y por que ese orden

Los tres transformadores se aplican en un orden que no es intercambiable:

1. **`AurumImputer`** primero, porque mientras la ley traiga el centinela `-1.0` cualquier
   promedio posterior queda contaminado. Convierte el valor especial en faltante, imputa con la
   mediana de las vecinas de los ultimos siete dias del mismo frente y tipo de mineral, y marca
   con `flag_imputed` las filas cuya ventana no llega a cinco lecturas.
2. **`AurumShiftEncoder`** despues, porque codifica `frente_id` y `equipo_id` usando la ley como
   objetivo y necesita esa ley ya limpia.
3. **`AurumFeatureBuilder`** al final, porque sus ventanas y rezagos se calculan sobre la ley
   tratada y sus banderas dependen de los umbrales del diccionario.

Cada uno cumple el mismo contrato —`fit`, `transform`, `fit_transform`— de modo que la
composicion es una lista y no un caso especial.

In [2]:
def ejecutar_pipeline(
    datos: pd.DataFrame,
    pasos: list[tuple[str, AurumTransformer, Callable[[pd.DataFrame], pd.Series] | None]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Aplica los transformadores en orden y devuelve el resultado y un resumen por paso.

    Cada paso puede traer una funcion que construya su objetivo a partir del marco que recibe.
    El codificador la usa para no aprender de leyes reconstruidas: una mediana de vecinas no es
    una medicion, y usarla para estimar la media de su propia categoria realimenta esa media
    consigo misma.

    Se resume en una tabla y no en prints sueltos para que quede constancia de cuanto agrega y
    cuanto tarda cada paso: es la informacion que se necesita cuando el pipeline crezca.
    """
    marco = datos
    resumen = []
    for nombre, transformador, objetivo in pasos:
        inicio = time.perf_counter()
        columnas_antes = set(marco.columns)
        marco = transformador.fit_transform(marco, objetivo(marco) if objetivo else None)
        resumen.append({
            "paso": nombre,
            "filas": len(marco),
            "columnas": marco.shape[1],
            "columnas_nuevas": ", ".join(sorted(set(marco.columns) - columnas_antes)) or "-",
            "segundos": round(time.perf_counter() - inicio, 2),
        })
    return marco, pd.DataFrame(resumen)


imputador = AurumImputer()
PASOS: list[tuple[str, AurumTransformer, Callable[[pd.DataFrame], pd.Series] | None]] = [
    ("AurumImputer", imputador, None),
    ("AurumShiftEncoder", AurumShiftEncoder(smoothing=10.0), imputador.objetivo_medido),
    ("AurumFeatureBuilder", AurumFeatureBuilder(ventana="7D", estadistico="media"), None),
]

final, resumen_pasos = ejecutar_pipeline(crudo, PASOS)

INFO    AurumImputer: ajuste de 50000 registros en 13 frentes: FR-C1-01, FR-C1-05, FR-C2-02, FR-C2-07, FR-N1-03, FR-N1-07, FR-N2-01, FR-N2-04, FR-N2-09, FR-S1-02, FR-S1-06, FR-S2-03, FR-S2-08


INFO    AurumImputer: historia de 47190 lecturas validas en 52 grupos


INFO    AurumImputer: transformacion de 50000 registros en 13 frentes: FR-C1-01, FR-C1-05, FR-C2-02, FR-C2-07, FR-N1-03, FR-N1-07, FR-N2-01, FR-N2-04, FR-N2-09, FR-S1-02, FR-S1-06, FR-S2-03, FR-S2-08


INFO    AurumImputer: 2810 centinelas, 2315 imputados con la mediana de la ventana, 495 marcados con flag_imputed por ventana con menos de 5 lecturas


INFO    AurumShiftEncoder: ajuste de 50000 registros en 13 frentes: FR-C1-01, FR-C1-05, FR-C2-02, FR-C2-07, FR-N1-03, FR-N1-07, FR-N2-01, FR-N2-04, FR-N2-09, FR-S1-02, FR-S1-06, FR-S2-03, FR-S2-08


...

INFO    AurumShiftEncoder: codificacion con leave-one-out de 50000 registros en 13 frentes: FR-C1-01, FR-C1-05, FR-C2-02, FR-C2-07, FR-N1-03, FR-N1-07, FR-N2-01, FR-N2-04, FR-N2-09, FR-S1-02, FR-S1-06, FR-S2-03, FR-S2-08


INFO    AurumFeatureBuilder: ajuste de 50000 registros en 13 frentes: FR-C1-01, FR-C1-05, FR-C2-02, FR-C2-07, FR-N1-03, FR-N1-07, FR-N2-01, FR-N2-04, FR-N2-09, FR-S1-02, FR-S1-06, FR-S2-03, FR-S2-08


INFO    AurumFeatureBuilder: historia de 50000 eventos para las features de ventana


INFO    AurumFeatureBuilder: transformacion de 50000 registros en 13 frentes: FR-C1-01, FR-C1-05, FR-C2-02, FR-C2-07, FR-N1-03, FR-N1-07, FR-N2-01, FR-N2-04, FR-N2-09, FR-S1-02, FR-S1-06, FR-S2-03, FR-S2-08


INFO    AurumFeatureBuilder: 9 features agregadas; 456 filas sin ventana suficiente


In [3]:
display(resumen_pasos)
print(f"columnas originales: {crudo.shape[1]}  ->  columnas finales: {final.shape[1]}")

,paso,filas,columnas,columnas_nuevas,segundos
0,AurumImputer,50000,19,flag_imputed,7.37
1,AurumShiftEncoder,50000,21,"equipo_id_target_enc, frente_id_target_enc",0.02
2,AurumFeatureBuilder,50000,30,"dias_desde_evento_previo, energia_especifica_p...",0.04


columnas originales: 18  ->  columnas finales: 30


El log de arriba es el que pide el enunciado: cada transformador deja constancia de cuantos
registros proceso y en que `frente_id`, tanto al ajustar como al transformar.

## 2. `AurumImputer` — el valor especial de la ley

El extracto trae 2810 lecturas con el centinela `-1.0`. La regla del enunciado es imputar con la
mediana de la ventana de siete dias del mismo frente y tipo de mineral, y marcar la fila cuando
esa ventana tenga menos de cinco lecturas validas.

In [4]:
centinelas = crudo[domain.COLUMNA_LEY].eq(domain.CENTINELA_LEY)
marcadas = final["flag_imputed"]
imputadas = centinelas & ~marcadas

print(f"centinelas en el extracto      : {int(centinelas.sum()):,} ({centinelas.mean() * 100:.2f}%)")
print(f"  imputadas con la mediana     : {int(imputadas.sum()):,}")
print(f"  marcadas con flag_imputed    : {int(marcadas.sum()):,} "
      f"({marcadas.sum() / centinelas.sum() * 100:.1f}% de los centinelas)")
print()
print("Las marcadas conservan la ley faltante, como pide el enunciado:")
print(f"  filas marcadas con ley nula  : {int(final.loc[marcadas, domain.COLUMNA_LEY].isna().sum()):,}")
print()
print("Reparto de los centinelas por turno, antes y despues:")
comparacion = pd.DataFrame({
    "centinelas": crudo.groupby("turno_cod")[domain.COLUMNA_LEY]
                       .apply(lambda s: s.eq(domain.CENTINELA_LEY).sum()),
    "marcadas": final.groupby("turno_cod")["flag_imputed"].sum(),
})
comparacion["imputadas"] = comparacion["centinelas"] - comparacion["marcadas"]
comparacion["pct_marcadas"] = (comparacion["marcadas"] / comparacion["centinelas"] * 100).round(1)
display(comparacion)

centinelas en el extracto      : 2,810 (5.62%)
  imputadas con la mediana     : 2,315
  marcadas con flag_imputed    : 495 (17.6% de los centinelas)

Las marcadas conservan la ley faltante, como pide el enunciado:
  filas marcadas con ley nula  : 495

Reparto de los centinelas por turno, antes y despues:


,centinelas,marcadas,imputadas,pct_marcadas
turno_cod,,,,
D1,257,51,206,19.8
D2,265,54,211,20.4
N1,278,37,241,13.3
N2,2010,353,1657,17.6


El 17.6% de los centinelas no se pudo imputar, y la razon es la intermitencia que documento el
EDA: un frente que vuelve despues de semanas apagado llega con la ventana de siete dias vacia.
La regla de los cinco registros no es una salvaguarda teorica, se activa en 495 filas.

## 3. `AurumShiftEncoder` — codificacion por objetivo

Codifica `frente_id` y `equipo_id` con leave-one-out. El objetivo que recibe no es la columna de
ley tal como sale del imputador sino `imputador.objetivo_medido(...)`, que deja en faltante las
2315 filas reconstruidas: una mediana de vecinas no es una medicion, y dejarla entrar a las
estadisticas de su propia categoria realimenta esa media consigo misma. El efecto medido en este
extracto es de milesimas de gramo por tonelada, pero la circularidad es real y evitarla cuesta
una linea.

El contraste entre las dos columnas codificadas es la verificacion mas util: el EDA mostro que el
frente separa la ley de 2.9 a 15.3 g/t y que el equipo es una etiqueta sin contenido, y el
encoding tiene que reflejarlo.

In [5]:
codificadas = [f"{columna}_target_enc" for columna in ("frente_id", "equipo_id")]
resumen_encoding = final[codificadas].describe().T[["mean", "std", "min", "max"]].round(3)
resumen_encoding["rango"] = (resumen_encoding["max"] - resumen_encoding["min"]).round(3)
display(resumen_encoding)

for columna in ("frente_id", "equipo_id"):
    codificada = f"{columna}_target_enc"
    correlacion = final[codificada].corr(final[domain.COLUMNA_LEY])
    print(f"{codificada:24s} corr con la ley: {correlacion:+.4f}")

,mean,std,min,max,rango
frente_id_target_enc,7.960,3.514,2.907,15.281,12.374
equipo_id_target_enc,7.998,0.054,7.910,8.065,0.155


frente_id_target_enc     corr con la ley: +0.9156
equipo_id_target_enc     corr con la ley: +0.0012


El encoding del frente recorre catorce puntos de ley y correlaciona 0.92 con el objetivo; el del
equipo se mueve dentro de un rango minimo alrededor de la media global y correlaciona 0.00. La
columna se codifica porque el enunciado la pide, y el resultado confirma lo que el EDA ya habia
medido.

## 4. `AurumFeatureBuilder` — las nueve features

Cuatro familias: historia de la ley en el frente, banderas de anomalia segun los umbrales del
diccionario, la bandera termica en el punto de quiebre medido en el extracto, y dos ratios
operacionales.

In [6]:
features = list(AurumFeatureBuilder.FEATURES)
descripcion = final[features].describe().T.round(3)
descripcion["nulos"] = final[features].isna().sum()
descripcion["pct_nulos"] = (final[features].isna().mean() * 100).round(2)
display(descripcion)

,count,mean,std,min,25%,50%,75%,max,nulos,pct_nulos
ley_ventana,49544.0,7.960,3.533,1.192,5.062,7.104,10.626,22.208,456,0.91
ley_n_ventana,50000.0,85.686,70.736,0.000,30.000,68.000,125.000,366.000,0,0.00
ley_lag_1,49986.0,7.960,3.847,0.870,4.791,7.316,10.572,25.760,14,0.03
dias_desde_evento_previo,49987.0,0.214,2.367,0.010,0.014,0.017,0.021,115.468,13,0.03
energia_especifica_proxy,50000.0,136375.355,45360.007,49404.246,106169.866,128292.683,156636.828,979704.000,0,0.00
sobretemperatura_por_rpm,50000.0,0.032,0.011,0.000,0.024,0.031,0.039,0.080,0,0.00


In [7]:
print("Cobertura de las banderas y su tasa de falla asociada:")
falla = final["falla_cod"].notna()
for bandera in ("flag_temp_riesgo", "flag_temp_apagado", "flag_vib_alerta"):
    activa = final[bandera]
    print(f"  {bandera:20s} n={int(activa.sum()):5d} ({activa.mean() * 100:5.2f}%) | "
          f"tasa de falla {falla[activa].mean() * 100:5.2f}%")
print(f"  {'linea base':20s} n={len(final):5d} (100.00%) | tasa de falla {falla.mean() * 100:5.2f}%")
print()
print("Correlacion de cada feature con la ley, sobre las filas con lectura valida:")
validas = final.loc[~final["flag_imputed"] & final[domain.COLUMNA_LEY].notna()]
correlaciones = (validas[[*features, domain.COLUMNA_LEY]]
                 .corr(numeric_only=True)[domain.COLUMNA_LEY].drop(domain.COLUMNA_LEY))
print(correlaciones.round(4).sort_values(ascending=False).to_string())

Cobertura de las banderas y su tasa de falla asociada:
  flag_temp_riesgo     n= 3602 ( 7.20%) | tasa de falla 22.68%
  flag_temp_apagado    n=  922 ( 1.84%) | tasa de falla 22.02%
  flag_vib_alerta      n=  140 ( 0.28%) | tasa de falla 17.14%
  linea base           n=50000 (100.00%) | tasa de falla  3.32%

Correlacion de cada feature con la ley, sobre las filas con lectura valida:
ley_ventana                 0.9117
ley_lag_1                   0.8384
ley_n_ventana               0.0175
flag_temp_riesgo            0.0049
flag_vib_alerta             0.0045
flag_temp_apagado           0.0006
dias_desde_evento_previo    0.0000
energia_especifica_proxy   -0.0000
sobretemperatura_por_rpm   -0.0028


`ley_ventana` alcanza 0.91 de correlacion con la ley, que es practicamente el techo teorico: el
EDA midio que la media del frente explica el 83% de la varianza y que el residuo es ruido blanco.
Las banderas reproducen exactamente las tasas de falla medidas en el analisis exploratorio, y los
ratios operacionales quedan en cero, como estaba previsto: se construyen por criterio de dominio
y su valor real se reporta en lugar de suponerse.

## 5. Verificacion de fuga de informacion

Las pruebas unitarias fijan este comportamiento sobre datos sinteticos. Aqui se comprueba sobre
las cincuenta mil filas reales, porque una fuga silenciosa es el unico error de este pipeline que
no se manifiesta como fallo sino como una metrica sospechosamente buena.

In [8]:
comparables = final.dropna(subset=["ley_lag_1", domain.COLUMNA_LEY])
correlacion_rezago = comparables["ley_lag_1"].corr(comparables[domain.COLUMNA_LEY])
antiguedades = final["dias_desde_evento_previo"].dropna()

assert correlacion_rezago < 0.95, "ley_lag_1 estaria replicando el valor de su propia fila"
assert (antiguedades > 0).all(), "hay antiguedades en cero: la historia estaria duplicada"
assert not final[domain.COLUMNA_LEY].eq(domain.CENTINELA_LEY).any(), "quedo un centinela sin tratar"
assert final.loc[final["flag_imputed"], domain.COLUMNA_LEY].isna().all(), \
    "una fila marcada como no imputable conserva un valor de ley"
assert list(final.columns[:crudo.shape[1]]) == list(crudo.columns), \
    "se altero el nombre o el orden de las columnas originales"

print(f"corr(ley_lag_1, ley del propio registro) : {correlacion_rezago:.4f}  "
      f"(el techo sano es la autocorrelacion del frente, 0.83)")
print(f"antiguedad minima entre eventos          : {antiguedades.min() * 24 * 60:.0f} minutos")
print(f"columnas originales intactas             : {crudo.shape[1]} de {crudo.shape[1]}")
print()
print("Todas las verificaciones de fuga pasaron.")

corr(ley_lag_1, ley del propio registro) : 0.8384  (el techo sano es la autocorrelacion del frente, 0.83)
antiguedad minima entre eventos          : 15 minutos
columnas originales intactas             : 18 de 18

Todas las verificaciones de fuga pasaron.


## 6. Tabla final consolidada

Primeras cincuenta filas del marco que entrega el pipeline: las dieciocho columnas originales sin
alterar, mas `flag_imputed`, las dos codificaciones y las nueve features.

In [9]:
print(f"forma final: {final.shape[0]:,} filas x {final.shape[1]} columnas")
print()
print("columnas agregadas por el pipeline:")
print(", ".join(columna for columna in final.columns if columna not in crudo.columns))

forma final: 50,000 filas x 30 columnas

columnas agregadas por el pipeline:
flag_imputed, frente_id_target_enc, equipo_id_target_enc, ley_ventana, ley_n_ventana, ley_lag_1, dias_desde_evento_previo, flag_temp_riesgo, flag_temp_apagado, flag_vib_alerta, energia_especifica_proxy, sobretemperatura_por_rpm


In [10]:
display(final.head(50))

,ts_opus_utc,frente_id,turno_cod,ley_au_gpT,ton_rom_acum,pres_hidraul_bar,rpm_corona,avance_mmin,agua_iny_lmin,vibracion_rms_ms2,temp_motor_c,op_id,equipo_id,falla_cod,prod_estimada_oz,tipo_mineral,sector_geol,flag_mant_prev,flag_imputed,frente_id_target_enc,equipo_id_target_enc,ley_ventana,ley_n_ventana,ley_lag_1,dias_desde_evento_previo,flag_temp_riesgo,flag_temp_apagado,flag_vib_alerta,energia_especifica_proxy,sobretemperatura_por_rpm
0,2023-07-01 00:16:00,FR-C1-05,N1,8.0641,273.320000,204.1,1042,1.288,65.40,4.000,72.0,OP-9BF3,EQ-SAND-11,NaN,64.4853,SUL,Cuerpo-Central,0,False,10.475621,7.946534,NaN,0.0,NaN,NaN,False,False,False,165118.167702,0.032630
1,2023-07-01 00:46:00,FR-C1-05,N1,10.9092,384.930000,188.6,932,1.528,47.24,3.525,59.9,OP-45C4,EQ-ATLAS-01,NaN,117.4591,OX,Cuerpo-Central,0,False,10.475016,8.059676,8.064100,1.0,8.0641,0.020833,False,False,False,115036.125654,0.023498
2,2023-07-01 01:14:00,FR-C1-05,N1,11.7270,372.210000,224.5,1352,1.740,48.81,3.622,71.0,OP-45C4,EQ-BOAR-05,NaN,127.7050,SUL,Cuerpo-Central,0,False,10.474842,7.946195,9.486650,2.0,10.9092,0.019444,False,False,False,174439.080460,0.024408
3,2023-07-01 01:37:00,FR-C1-05,N1,9.7253,349.540000,194.0,1208,1.899,23.81,7.271,76.7,OP-70EF,EQ-BOAR-05,NaN,99.4564,SUL,Cuerpo-Central,0,False,10.475268,7.946616,10.233433,3.0,11.7270,0.015972,False,False,False,123408.109531,0.032036
4,2023-07-01 02:00:00,FR-C1-05,N1,7.6662,318.730000,172.4,851,2.276,61.86,5.317,66.6,OP-C74D,EQ-ATLAS-01,NaN,71.4886,SUL,Cuerpo-Central,0,False,10.475706,8.060382,10.106400,4.0,9.7253,0.015972,False,False,False,64460.632689,0.033608
5,2023-07-01 02:32:00,FR-C1-05,N1,8.8637,163.390000,209.9,1294,1.493,57.37,1.908,66.5,OP-70EF,EQ-SAND-10,NaN,38.6466,MIX,Cuerpo-Central,0,False,10.475451,7.913339,9.618360,5.0,7.6662,0.022222,False,False,False,181922.705961,0.022025
6,2023-07-01 02:48:00,FR-C1-05,N1,11.2502,382.260000,183.7,1127,1.502,48.90,9.484,63.2,OP-1679,EQ-ATLAS-04,NaN,120.2899,OX,Cuerpo-Central,0,False,10.474943,8.062798,9.492583,6.0,8.8637,0.011111,False,False,False,137836.151798,0.022360
7,2023-07-01 03:16:00,FR-C1-05,N1,10.4327,385.970000,183.8,1024,1.400,45.78,4.545,80.8,OP-E4DA,EQ-SAND-09,NaN,112.6312,OX,Cuerpo-Central,0,False,10.475117,8.035987,9.743671,7.0,11.2502,0.019444,False,False,False,134436.571429,0.041797
8,2023-07-01 03:35:00,FR-C1-05,N1,8.2693,271.940000,202.5,1045,1.537,36.28,4.507,78.6,OP-8F14,EQ-SAND-11,NaN,62.9000,OX,Cuerpo-Central,0,False,10.475578,7.946491,9.829800,8.0,10.4327,0.013194,False,False,False,137678.919974,0.038852
9,2023-07-01 03:51:00,FR-C1-05,N1,9.5778,188.080000,204.3,1148,1.792,48.08,2.892,78.9,OP-AAB3,EQ-ATLAS-03,NaN,50.3871,OX,Cuerpo-Central,0,False,10.475299,8.020912,9.656411,9.0,8.2693,0.011111,False,False,False,130879.687500,0.035627


## 7. Lo que queda verificado

- Los tres transformadores comparten el contrato `fit` / `transform` / `fit_transform` y dejan en
  el log los frentes procesados en cada paso.
- El centinela de la ley desaparece del marco final: 2315 filas imputadas con la mediana de su
  ventana y 495 marcadas por ventana insuficiente, con la ley en faltante.
- La codificacion por objetivo separa el frente y deja al equipo en la media global, que es lo que
  el analisis exploratorio anticipaba.
- Las nueve features estan construidas sobre el pasado estricto: ninguna ventana, rezago ni
  antiguedad ve la propia fila, y las cuatro verificaciones de fuga pasan sobre el extracto real.
- Ninguna columna original cambio de nombre, tipo ni orden.